In [1]:
import pandas as pd
import numpy as np


In [3]:
interactions = pd.read_csv("../data/interactions.csv")
products = pd.read_csv("../data/products.csv")
users = pd.read_csv("../data/users.csv")


In [5]:
interactions.head()


,user_id,product_id,event_type,timestamp
0,349,73,add_to_cart,2025-09-08 17:25:19.936687
1,320,10,add_to_cart,2025-07-13 17:25:19.936687
2,421,61,click,2025-07-21 17:25:19.936687
3,468,50,add_to_cart,2025-06-30 17:25:19.936687
4,49,122,click,2025-12-12 17:25:19.936687


In [7]:
user_features = interactions.groupby("user_id").agg(
    total_interactions=("event_type", "count")
).reset_index()


In [9]:
purchase_counts = interactions[interactions["event_type"] == "purchase"] \
    .groupby("user_id").size().reset_index(name="purchase_count")

user_features = user_features.merge(
    purchase_counts, on="user_id", how="left"
)

user_features["purchase_count"] = user_features["purchase_count"].fillna(0)


In [11]:
user_category = interactions.merge(products, on="product_id")

preferred_category = (
    user_category.groupby(["user_id", "category"])
    .size()
    .reset_index(name="count")
    .sort_values(["user_id", "count"], ascending=False)
    .drop_duplicates("user_id")
)

user_features = user_features.merge(
    preferred_category[["user_id", "category"]],
    on="user_id",
    how="left"
)

user_features.rename(columns={"category": "preferred_category"}, inplace=True)


In [13]:
user_features.head()


,user_id,total_interactions,purchase_count,preferred_category
0,1,38,8.0,Electronics
1,2,33,5.0,Beauty
2,3,31,7.0,Fashion
3,4,32,4.0,Fashion
4,5,31,4.0,Beauty


In [15]:
products["price_norm"] = (
    products["price"] - products["price"].min()
) / (products["price"].max() - products["price"].min())

products["margin_norm"] = (
    products["profit_margin"] - products["profit_margin"].min()
) / (products["profit_margin"].max() - products["profit_margin"].min())


In [17]:
product_popularity = interactions.groupby("product_id").size().reset_index(name="popularity")

products = products.merge(product_popularity, on="product_id", how="left")
products["popularity"] = products["popularity"].fillna(0)


In [19]:
products.head()


,product_id,category,price,profit_margin,inventory_level,price_norm,margin_norm,popularity
0,1,Fashion,1135,0.38,58,0.195030,0.942857,64
1,2,Fashion,4537,0.17,301,0.905408,0.342857,59
2,3,Home,201,0.09,391,0.000000,0.114286,65
3,4,Beauty,3048,0.22,333,0.594487,0.485714,72
4,5,Beauty,4244,0.14,87,0.844226,0.257143,69


In [21]:
rating_map = {
    "click": 1,
    "add_to_cart": 3,
    "purchase": 5
}

interactions["rating"] = interactions["event_type"].map(rating_map)


In [23]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="product_id",
    values="rating",
    aggfunc="mean"
).fillna(0)


In [25]:
products["content"] = (
    products["category"] + " " +
    products["price"].astype(str)
)


In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
product_tfidf = tfidf.fit_transform(products["content"])
